# Social Media Comment Scraper
Topik: Pembatasan Sosial Media dan Game Roblox untuk Anak di Bawah 16 Tahun


## Load Library dan API

In [1]:
import os
import re
import time
import pandas as pd
from dotenv import load_dotenv

# Google / YouTube API
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


load_dotenv(dotenv_path=".env", override=True)

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY")

keys = {
    "YOUTUBE_API_KEY" : YOUTUBE_API_KEY,
}
for nama, nilai in keys.items():
    status = "BERHASIL" if nilai else " BELUM DIISI"
    print(f"{nama} {status} ")




YOUTUBE_API_KEY BERHASIL 


## Scraping Komentar melalui Link Postingan

### Daftar Link Postingan
Berisi link YouTube (Video & Short) dan TikTok dari media-media yang telah ditentukan.

In [2]:
# --- YouTube VIDEO ---
YOUTUBE_VIDEO_LINKS = [
    {"source": "Tribunnews", "url": "https://youtu.be/2YHfX7PzVNM"},
    {"source": "CNN", "url": "https://www.youtube.com/watch?v=p8p-sAq_hhI"},
    {"source": "TVOne", "url": "https://youtu.be/R1hQwelEGvU"},
    {"source": "MetroTV", "url": "https://youtu.be/tuaZNgn2rcc"},
    {"source": "Berita Satu", "url": "https://youtu.be/LFfRhLKHdJA"},
]

# --- YouTube SHORT ---
YOUTUBE_SHORT_LINKS = [
    {"source": "Kumparan", "url": "https://youtube.com/shorts/hzDavp3DQo4"},
    {"source": "Kompas", "url": "https://youtube.com/shorts/2BdXibzPPyc"},
    {"source": "Bisniscom", "url": "https://youtube.com/shorts/TdyE62mSXx0"},
]

# --- Output paths ---
OUTPUT_YT_LINK = "youtube_link_comments.csv"
OUTPUT_YTS_LINK = "youtube_shorts_link_comments.csv"


print("Daftar link postingan dimuat:")
print(f"YouTube Video : {len(YOUTUBE_VIDEO_LINKS)} link")
print(f"YouTube Short : {len(YOUTUBE_SHORT_LINKS)} link")



Daftar link postingan dimuat:
YouTube Video : 5 link
YouTube Short : 3 link



### Scraping YouTube Video — dari Link Spesifik


In [ ]:
def ekstrak_video_id(url):
    """Ekstrak 11-karakter video ID dari berbagai format URL YouTube."""
    patterns = [
        r"youtu\.be/([\w-]{11})",
        r"youtube\.com/watch\?v=([\w-]{11})",
        r"youtube\.com/shorts/([\w-]{11})",
        r"youtube\.com/embed/([\w-]{11})",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return None


def ekstrak_judul_youtube(youtube, vid_id):
    try:
        resp = youtube.videos().list(part="snippet", id=vid_id).execute()
        if "items" in resp and resp["items"]:
            return resp["items"][0]["snippet"]["title"]
    except Exception:
        pass
    return ""

def scrape_yt_dari_link(youtube, link_list, platform_label="YouTube", max_komentar=10000):
    
    semua = []
    for item in link_list:
        source = item["source"]
        url = item["url"]
        vid_id = ekstrak_video_id(url)
        vid_title = ekstrak_judul_youtube(youtube, vid_id)
        if not vid_id:
            print(f"Tidak bisa ekstrak video ID dari: {url}")
            continue
        print(f"[{source}] video_id={vid_id}")
        next_page = None
        count = 0
        try:
            while count < max_komentar:
                resp = youtube.commentThreads().list(
                    part="snippet",
                    videoId=vid_id,
                    maxResults=min(100, max_komentar - count),
                    pageToken=next_page,
                    textFormat="plainText"
                ).execute()
                for i in resp.get("items", []):
                    snip = i["snippet"]["topLevelComment"]["snippet"]
                    semua.append({
                        "platform" : platform_label,
                        "video_id" : vid_id,
                        "video_title" : vid_title,
                        "author" : snip.get("authorDisplayName", ""),
                        "comment" : snip.get("textDisplay", ""),
                        "published_at": snip.get("publishedAt", ""),
                        "reply" : 0,
                        "like_comment": snip.get("likeCount", 0)
                    })
                    count += 1
                    if count >= max_komentar:
                        break
                next_page = resp.get("nextPageToken")
                if not next_page:
                    break
            print(f"{count} komentar")
        except HttpError as e:
            if e.resp.status == 403:
                print(f"Komentar dinonaktifkan atau kuota habis.")
            else:
                print(f"HttpError: {e}")
        time.sleep(0.5)
    return semua

print("=" * 60)
print("YOUTUBE VIDEO — SCRAPING DARI LINK")
print("=" * 60)

youtube_client = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

hasil_yt_link = scrape_yt_dari_link(
    youtube_client, YOUTUBE_VIDEO_LINKS,
    platform_label="YouTube", max_komentar=500
)

df_yt_link = pd.DataFrame(hasil_yt_link)
if not df_yt_link.empty:
    df_yt_link.to_csv(OUTPUT_YT_LINK, index=False, encoding="utf-8-sig")
    print(f"\n Disimpan: {OUTPUT_YT_LINK} | Total: {len(df_yt_link)} komentar")
    display(df_yt_link.head(3))
else:
    print("Tidak ada data komentar YouTube Video (link).")


YOUTUBE VIDEO — SCRAPING DARI LINK
[Tribunnews] video_id=2YHfX7PzVNM
67 komentar
[CNN] video_id=p8p-sAq_hhI
487 komentar
[TVOne] video_id=R1hQwelEGvU
287 komentar
[MetroTV] video_id=tuaZNgn2rcc
200 komentar
[Berita Satu] video_id=LFfRhLKHdJA
86 komentar

 Disimpan: youtube_link_comments.csv | Total: 1127 komentar


,platform,video_id,video_title,author,comment,published_at,reply,like_comment
0,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@AkmalCakep-nh6du,Jangan batasi akun saya banyak konten gta sama Hp,2026-03-27T15:28:30Z,0,0
1,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@IbnuBungo-d2y,Masalah nya yang di pake hp orang tua nya\nGak...,2026-03-16T12:21:29Z,0,0
2,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@ULTRAGAMEXS52,Bu mereka susah dipahami,2026-03-15T02:12:15Z,0,0


---
### Scraping YouTube Short — dari Link Spesifik
> YouTube Shorts diperlakukan sama dengan video biasa oleh API.  
> Hasil disimpan ke `output/youtube_shorts_link_comments.csv`

In [ ]:
print("=" * 60)
print("YOUTUBE SHORT — SCRAPING DARI LINK")
print("=" * 60)


hasil_yts_link = scrape_yt_dari_link(
    youtube_client, YOUTUBE_SHORT_LINKS,
    platform_label="YouTube Short", max_komentar=10000
)

df_yts_link = pd.DataFrame(hasil_yts_link)
if not df_yts_link.empty:
    df_yts_link.to_csv(OUTPUT_YTS_LINK, index=False, encoding="utf-8-sig")
    print(f"\n Disimpan: {OUTPUT_YTS_LINK} | Total: {len(df_yts_link)} komentar")
    display(df_yts_link.head(3))
else:
    print("Tidak ada data komentar YouTube Short (link).")


YOUTUBE SHORT — SCRAPING DARI LINK
[Kumparan] video_id=hzDavp3DQo4
3134 komentar
[Kompas] video_id=2BdXibzPPyc
96 komentar
[Bisniscom] video_id=TdyE62mSXx0
1078 komentar

 Disimpan: youtube_shorts_link_comments.csv | Total: 4308 komentar


,platform,video_id,video_title,author,comment,published_at,reply,like_comment
0,YouTube Short,hzDavp3DQo4,"Komdigi: Mulai 28 Maret, Anak di Bawah 16 Tahu...",@Dhika-zt5lf,Daripada memblokir mending dikasih peringatan ...,2026-03-31T06:29:13Z,0,0
1,YouTube Short,hzDavp3DQo4,"Komdigi: Mulai 28 Maret, Anak di Bawah 16 Tahu...",@kasper-u5f,Coba aku jadi presiden pasti aku bakal Gak ke ...,2026-03-31T06:16:54Z,0,0
2,YouTube Short,hzDavp3DQo4,"Komdigi: Mulai 28 Maret, Anak di Bawah 16 Tahu...",@LindaJuniarti-w6s,aku gak setuju 😡,2026-03-31T05:43:58Z,0,0


## Scraping Komentar melalui Keyword

### Keyword & Pengaturan Scraping


In [ ]:

YOUTUBE_KEYWORDS = [
    "pembatasan sosial media anak",
    "larangan medsos anak 16 tahun",
    "aturan baru medsos anak",
    "blokir game anak di bawah umur",
    "bahaya sosmed game anak",
]
YOUTUBE_MAX_VIDEOS = 20
YOUTUBE_MAX_COMMENTS_PER_VIDEO = 1000

OUTPUT_YOUTUBE = "youtube_comments_keyword.csv"


print("Konfigurasi berhasil dimuat untuk audiens Indonesia!")
print(f"YouTube : {len(YOUTUBE_KEYWORDS)} keyword, maks {YOUTUBE_MAX_VIDEOS} video")



Konfigurasi berhasil dimuat untuk audiens Indonesia!
YouTube : 5 keyword, maks 20 video


In [6]:
def youtube_cari_video(youtube, keyword, max_results):
    print(f"Keyword: '{keyword}'")
    videos = []
    next_page_token = None
    
    try:
        while len(videos) < max_results:
            fetch_count = min(50, max_results - len(videos))
            response = youtube.search().list(
                q=keyword, 
                part="snippet", 
                type="video",
                maxResults=fetch_count, 
                order="relevance",
                pageToken=next_page_token
            ).execute()
            
            for i in response.get("items", []):
                vid_id = i.get("id", {}).get("videoId")
                if vid_id:
                    videos.append({
                        "video_id": vid_id, 
                        "video_title": i["snippet"]["title"]
                    })
                    
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
    except Exception as e:
        print(f"Error saat mencari video: {e}")
        
    print(f"{len(videos)} video ditemukan")
    return videos[:max_results]

def youtube_ambil_komentar(youtube, video_id, video_title, max_komentar):
    komentar_list = []
    next_page_token = None
    try:
        while len(komentar_list) < max_komentar:
            fetch_count = min(100, max_komentar - len(komentar_list))
            response = youtube.commentThreads().list(
                part="snippet", 
                videoId=video_id,
                maxResults=fetch_count, 
                pageToken=next_page_token,
                textFormat="plainText"
            ).execute()
            
            for item in response.get("items", []):
                thread_snippet = item["snippet"]
                snip = thread_snippet["topLevelComment"]["snippet"]
                
                komentar_list.append({
                    "platform" : "YouTube",
                    "video_id" : video_id,
                    "video_title" : video_title,
                    "author" : snip.get("authorDisplayName", ""),
                    "comment" : snip.get("textDisplay", ""),
                    "published_at": snip.get("publishedAt", ""),
                    "reply" : thread_snippet.get("totalReplyCount", 0),
                    "like_comment": snip.get("likeCount", 0)
                })
                
                if len(komentar_list) >= max_komentar:
                    break
                    
            next_page_token = response.get("nextPageToken")
            if not next_page_token:
                break
    except HttpError as e:
        if e.resp.status == 403:
            print(f"Komentar dinonaktifkan: {video_title[:50]}...")
        elif e.resp.status == 404:
            print(f"Video tidak ditemukan (mungkin dihapus/private): {video_title[:50]}...")
        else:
            print(f"Error API: {e}")
    except Exception as e:
        print(f"Terjadi kesalahan: {e}")
        
    return komentar_list

print("="*60)
print("YOUTUBE SCRAPER (KEYWORD)")
print("="*60)

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)
semua_komentar_yt = []
sudah_diambil_yt = set()

for kw in YOUTUBE_KEYWORDS:
    videos = youtube_cari_video(youtube, kw, YOUTUBE_MAX_VIDEOS)
    for v in videos:
        if v["video_id"] in sudah_diambil_yt:
            continue
        sudah_diambil_yt.add(v["video_id"])
        
        print(f"{v['video_title'][:60]}...")
        km = youtube_ambil_komentar(youtube, v["video_id"], v["video_title"], YOUTUBE_MAX_COMMENTS_PER_VIDEO)
        semua_komentar_yt.extend(km)
        print(f"{len(km)} komentar")
        time.sleep(0.5)

df_youtube = pd.DataFrame(semua_komentar_yt)
if not df_youtube.empty:
    df_youtube.to_csv(OUTPUT_YOUTUBE, index=False, encoding="utf-8-sig")
    print(f"\nDisimpan: {OUTPUT_YOUTUBE} | Total: {len(df_youtube)} komentar")
    display(df_youtube.head(3))
else:
    print("Tidak ada komentar yang ditemukan untuk YouTube.")


YOUTUBE SCRAPER (KEYWORD)
Keyword: 'pembatasan sosial media anak'
25 video ditemukan
Pemerintah Resmikan Pembatasan Akses Digital Untuk Anak Diba...
150 komentar
Pembatasan Medsos Untuk Anak di Bawah 16 Tahun | Kabar Hari ...
49 komentar
Pembatasan Media Sosial Anak Efektif 28 Maret 2026...
22 komentar
[HEADLINE NEWS, 06/03] Penerapan Batasan Akses Media Sosial ...
200 komentar
Pembatasan Medsos untuk Akun Anak di Bawah 16 Tahun Dinonakt...
85 komentar
Menkomdigi Resmi Batasi Medsos, Anak di Bawah 16 Tahun Dilar...
287 komentar
Mulai Maret 2026, Pemerintah Bakal Batasi Akses Medsos untuk...
21 komentar
Medsos Anak Dibatasi, Pengamat: Kebijakan Ini Sudah Lama Din...
7 komentar
Komdigi: Mulai 28 Maret, Anak di Bawah 16 Tahun Dilarang Pun...
1000 komentar
Pemerintah Batasi Akun Medsos Anak di Bawah 16 Tahun...
487 komentar
Wacana Pembatasan Usia Pengguna Medsos, Lindungi Anak di Rua...
23 komentar
Indonesia No. 1 dalam Pembatasan Media Sosial Anak - [Priori...
1 komentar
Aturan Baru! Komd

,platform,video_id,video_title,author,comment,published_at,reply,like_comment
0,YouTube,0Sa4S97M-uM,Pemerintah Resmikan Pembatasan Akses Digital U...,@supriyatnosupriyatno7726,Pak kominfo..sy usul ..sy minta masa berlaku q...,2026-03-30T23:22:32Z,0,0
1,YouTube,0Sa4S97M-uM,Pemerintah Resmikan Pembatasan Akses Digital U...,@Jinangachatuber,Gak peduli gue 🗿,2026-03-30T06:11:27Z,0,1
2,YouTube,0Sa4S97M-uM,Pemerintah Resmikan Pembatasan Akses Digital U...,@rizky11-n4u,Roblox di brokoli tapi judol judi engak ujian ...,2026-03-30T05:10:55Z,0,0


## Gabungkan Data Scrapping

In [ ]:


print("=" * 60)
print("📁 MENGGABUNGKAN SELURUH HASIL SCRAPING SECARA KESELURUHAN (TANPA FILTER DUPLIKAT)")
print("=" * 60)


semua_file_csv = [
    "youtube_link_comments.csv",
    "youtube_shorts_link_comments.csv",
    "youtube_comments_keyword.csv",
    
]

dataframes = []

for path in semua_file_csv:
    if os.path.exists(path):
        # Membaca file
        temp_df = pd.read_csv(path)
        if not temp_df.empty:
            dataframes.append(temp_df)
            print(f"✔️ Berhasil memuat: {path} ({len(temp_df)} baris)")
    else:
        print(f"⚠️ File tidak ditemukan: {path}")

# Menggabungkan data tanpa drop_duplicates
if dataframes:
    # Menggabungkan semua dataset menjadi satu DataFrame
    df_final = pd.concat(dataframes, ignore_index=True)
    
    # Simpan sebagai master-data
    output_akhir = "all_scraped_comments_final.csv"
    df_final.to_csv(output_akhir, index=False, encoding="utf-8-sig")
    
    print("\n✅ PENGGABUNGAN DATA BERHASIL")
    print(f"Total komentar (Data Mentah) : {len(df_final)}")
    print(f"File master tersimpan di     : {output_akhir}")
    
    print("\nDistribusi Data per Platform:")
    print(df_final['platform'].value_counts())
    
    print("\nSampel Data Utama:")
    display(df_final.head(5))
else:
    print("\n❌ Terjadi kesalahan: Tidak ada satupun file CSV hasil scraping di folder 'output'.")


📁 MENGGABUNGKAN SELURUH HASIL SCRAPING SECARA KESELURUHAN (TANPA FILTER DUPLIKAT)
✔️ Berhasil memuat: youtube_link_comments.csv (1127 baris)
✔️ Berhasil memuat: youtube_shorts_link_comments.csv (4308 baris)
✔️ Berhasil memuat: youtube_comments_keyword.csv (20464 baris)

✅ PENGGABUNGAN DATA BERHASIL
Total komentar (Data Mentah) : 25899
File master tersimpan di     : all_scraped_comments_final.csv

Distribusi Data per Platform:
platform
YouTube          21591
YouTube Short     4308
Name: count, dtype: int64

Sampel Data Utama:


,platform,video_id,video_title,author,comment,published_at,reply,like_comment
0,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@AkmalCakep-nh6du,Jangan batasi akun saya banyak konten gta sama Hp,2026-03-27T15:28:30Z,0,0
1,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@IbnuBungo-d2y,Masalah nya yang di pake hp orang tua nya\nGak...,2026-03-16T12:21:29Z,0,0
2,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@ULTRAGAMEXS52,Bu mereka susah dipahami,2026-03-15T02:12:15Z,0,0
3,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@wilsenwilsen1473,Kenapa ga jelas kita bukan negara seperti erop...,2026-03-13T05:45:30Z,0,0
4,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@CureCharm_VermillionStar,"ban semua medsos kecuali CHATGPT , kenapa aku ...",2026-03-12T13:41:03Z,0,0


In [ ]:
import pandas as pd


file_path = "all_scraped_comments_final.csv"
df_master = pd.read_csv(file_path)


print("========== INFORMASI STRUKTUR DATA ==========")
df_master.info()


print("\n========== SAMPEL 5 DATA ==========")
display(df_master.head(5))


print("\n========== JUMLAH DATA KOSONG (Komentar Kosong) ==========")
print(df_master.isnull().sum())


========== INFORMASI STRUKTUR DATA ==========
<class 'pandas.DataFrame'>
RangeIndex: 25899 entries, 0 to 25898
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   platform      25899 non-null  str  
 1   video_id      25899 non-null  str  
 2   video_title   25899 non-null  str  
 3   author        25899 non-null  str  
 4   comment       25899 non-null  str  
 5   published_at  25899 non-null  str  
 6   reply         25899 non-null  int64
 7   like_comment  25899 non-null  int64
dtypes: int64(2), str(6)
memory usage: 1.6 MB

========== SAMPEL 5 DATA ==========


,platform,video_id,video_title,author,comment,published_at,reply,like_comment
0,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@AkmalCakep-nh6du,Jangan batasi akun saya banyak konten gta sama Hp,2026-03-27T15:28:30Z,0,0
1,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@IbnuBungo-d2y,Masalah nya yang di pake hp orang tua nya\nGak...,2026-03-16T12:21:29Z,0,0
2,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@ULTRAGAMEXS52,Bu mereka susah dipahami,2026-03-15T02:12:15Z,0,0
3,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@wilsenwilsen1473,Kenapa ga jelas kita bukan negara seperti erop...,2026-03-13T05:45:30Z,0,0
4,YouTube,2YHfX7PzVNM,"Indonesia Batasi Penggunaan Medsos, Anak 13-16...",@CureCharm_VermillionStar,"ban semua medsos kecuali CHATGPT , kenapa aku ...",2026-03-12T13:41:03Z,0,0



========== JUMLAH DATA KOSONG (Komentar Kosong) ==========
platform        0
video_id        0
video_title     0
author          0
comment         0
published_at    0
reply           0
like_comment    0
dtype: int64
